expanding/exploring 2.1 stuff : )
also testing out mlflow for tracking because its getting hectic

In [18]:
import pandas as pd
import numpy as np
from factor_analyzer import FactorAnalyzer
from src.features import feature_encoder, raw_data_loader
import os 

In [19]:
os.environ["MLFLOW_EXPERIMENT_NAME"] = "2.1-jp-feature-engineering"

In [20]:
df = raw_data_loader.load_and_clean_raw("../")

df = feature_encoder.encode_features(df)

In [21]:
df.head()

,age,bmi,sex,sw_9am_start_diff,sw_5pm_end_diff,nasal_congestion_stuffiness_nose,nasal_blockage_obstr_nose,troub_brth_nose,troub_slp_nose,not_enough_air_excercise_nose,...,pulmonary_problem_other_mdhx,chronic_obstructive_pulmonary_disease_mdhx,asthma_mdhx,cardiovascular_problem_other_mdhx,congestive_heart_failure_mdhx,hypertension_mdhx,oophorectomy_bilateral_mdhx,ahi,dream_recall_frequency_infrequent,dream_recall_frequency_rarely_or_never
0,58.0,30.7,1.0,-1.00,0.0,1.0,0.0,0.0,2.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.3,0.0,1.0
1,30.0,29.4,1.0,-1.00,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.4,1.0,0.0
2,30.0,25.8,1.0,-2.00,2.0,0.0,0.0,0.0,4.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
3,42.0,26.8,0.0,-3.00,3.0,1.0,0.0,0.0,3.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.9,0.0,1.0
4,36.0,45.2,0.0,-0.75,0.5,3.0,1.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.3,0.0,1.0


In [22]:
import mlflow

mlflow.set_tracking_uri("file:///Users/jack/Repos/apnea-predictor/mlruns")
mlflow.set_experiment(os.getenv("MLFLOW_EXPERIMENT_NAME", "default_experiment"))

<Experiment: artifact_location='file:///Users/jack/Repos/apnea-predictor/mlruns/620249075279939753', creation_time=1776455710940, experiment_id='620249075279939753', last_update_time=1776455710940, lifecycle_stage='active', name='2.1-jp-feature-engineering', tags={}, workspace='default'>

In [23]:
from src.features.transformers import Factor_Analyzer_Transformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from src.utils.data_utils import convert_ahi
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, f1_score, classification_report
from mlflow.models.signature import infer_signature


X = df.drop(columns=["ahi"])
y = convert_ahi(df["ahi"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

train_data = pd.concat([X_train, y_train], axis=1)

train_dataset = mlflow.data.from_pandas(
    train_data, name="2.1-sleep-apnea-train-raw", targets="ahi"
)

test_dataset = mlflow.data.from_pandas(
    pd.concat([X_test, y_test], axis=1), name="2.1-sleep-apnea-test-raw", targets="ahi"
)

with mlflow.start_run():
    mlflow.log_input(train_dataset, context="train")
    mlflow.log_input(test_dataset, context="test")
    params = {
        "objective": "binary:logistic",
        "max_depth": 6,
        "learning_rate": 0.1,
        "n_estimators": 100,
        "random_state": 42,
        "test_size": 0.2,
    }
    mlflow.log_params(params)
    model = XGBClassifier(**params)
    model = model.fit(X_train, y_train)
    sig = infer_signature(X_train, model.predict(X_train))

    mlflow.set_tag("model_type", "xgboost")
    mlflow.xgboost.log_model(
        xgb_model=model, name="xgb_model", model_format="json", signature=sig
    )

    y_pred = model.predict(X_test)
    mlflow.log_metric("test_accuracy", np.mean(y_pred == y_test))
    mlflow.log_metric("test_auc", roc_auc_score(y_test, y_pred))
    mlflow.log_metric("test_f1", f1_score(y_test, y_pred))
    mlflow.log_text(classification_report(y_test, y_pred), "classification_report.txt")

/Users/jack/Repos/apnea-predictor/conda_env/lib/python3.14/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
/Users/jack/Repos/apnea-predictor/conda_env/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [15:59:32] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "test_size" } 

In [24]:
from src.eval import evaluate_model
fa_transformer = Factor_Analyzer_Transformer(n_factors=18, rotation="promax")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

fa_transformer.fit(X_train)

X_fa = fa_transformer.transform(X)

df_fa = pd.DataFrame(X_fa, columns=[f"Factor_{i+1}" for i in range(X_fa.shape[1])])
df_fa["ahi"] = y.values

X_train_fa = fa_transformer.transform(X_train)
X_test_fa = fa_transformer.transform(X_test)

train_dataset_fa = mlflow.data.from_pandas(
    pd.concat([X_train_fa, y_train], axis=1), name="2.2-sleep-apnea-train-fa-promax", targets="ahi"
)

test_dataset_fa = mlflow.data.from_pandas(
    pd.concat([X_test_fa, y_test], axis=1), name="2.2-sleep-apnea-test-fa-promax", targets="ahi"
)


with mlflow.start_run():
    mlflow.log_input(train_dataset_fa, context="train")
    mlflow.log_input(test_dataset_fa, context="test")
    params = {
        "objective": "binary:logistic",
        "max_depth": 6,
        "learning_rate": 0.1,
        "n_estimators": 100,
        "random_state": 42,
        "test_size": 0.2,
    }
    mlflow.log_params(params)
    model = XGBClassifier(**params)
    model = model.fit(X_train_fa, y_train)
    sig = infer_signature(X_train_fa, model.predict(X_train_fa))

    mlflow.set_tag("model_type", "xgboost")
    mlflow.xgboost.log_model(
        xgb_model=model, name="xgb_model", model_format="json", signature=sig
    )
    y_pred = model.predict(X_test_fa)
    mlflow.log_metric("test_accuracy", np.mean(y_pred == y_test))
    mlflow.log_metric("test_auc", roc_auc_score(y_test, y_pred))
    mlflow.log_metric("test_f1", f1_score(y_test, y_pred))
    mlflow.log_text(classification_report(y_test, y_pred), "classification_report.txt")

/Users/jack/Repos/apnea-predictor/conda_env/lib/python3.14/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
/Users/jack/Repos/apnea-predictor/conda_env/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [15:59:37] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "test_size" } 

In [25]:
fa_transformer.get_top_loadings()

Top loadings for Factor_1:
fati_interferes_with_phys_function_fss       1.031793
fati_prevents_sustained_phys_func_fss        1.019358
fati_interferes_with_responsibilities_fss    0.977720
fati_top_three_disabling_sympts_fss          0.957607
fati_causes_freq_probs_for_me_fss            0.928218
i_am_easily_fati_fss                         0.924449
fati_interferes_with_work_fam_social_fss     0.858591
lower_motivation_when_fati_fss               0.644888
Name: Factor_1, dtype: float64


Top loadings for Factor_2:
diff_staying_asleep_isi                      0.930820
isi_total_score                              0.929114
troub_slp_nose                               0.810117
how_satisfied_with_curr_sleep_pattern_isi    0.794006
slp_quality_sw                               0.713055
diff_falling_asleep_isi                      0.672223
diff_falling_asleep_isq                      0.633961
frequent_wakenings_map                       0.632612
Name: Factor_2, dtype: float64


Top loadings for

In [26]:
df_fa_alt = df.copy()

# Drop columns: 'ess_total_score', 'isi_total_score'
df_fa_alt = df_fa_alt.drop(columns=['ess_total_score', 'isi_total_score'])

In [28]:
def create_versioned_dataset(data, version, base_name="sleep-apnea-data", tags= None):
    """Create a versioned dataset with metadata."""

    dataset = mlflow.data.from_pandas(
        data,
        source=f"data_pipeline_v{version}",
        name=f"{base_name}-v{version}",
        targets="ahi",
    )

    with mlflow.start_run(run_name=f"Dataset_Version_{version}"):
        mlflow.log_inputs([dataset], contexts=["versioning"], tags_list=[tags])

        # Log version metadata
        mlflow.log_params({
            "dataset_version": version,
            "data_size": data.shape,
            "features_count": data.shape[1] - 1,
            "target_distribution": data["ahi"].value_counts().to_dict(),
        })

        # Log data quality metrics
        mlflow.log_metrics({
            "missing_values_pct": (data.isnull().sum().sum() / data.size) * 100,
            "duplicate_rows": data.duplicated().sum(),
            "target_balance": data["ahi"].std(),
        })

    return dataset


In [30]:
"""
varimax fa 
"""
fa_transformer_var = Factor_Analyzer_Transformer(n_factors=18, rotation="varimax")

fa_transformer_var.fit(X_train)

X = fa_transformer_var.transform(X)

df_fa_var = pd.DataFrame(X, columns=[f"Factor_{i+1}" for i in range(X.shape[1])])
df_fa_var["ahi"] = y.values


In [32]:
fa_transformer_alt = Factor_Analyzer_Transformer(n_factors=18, rotation="promax")

X_alt = df_fa_alt.drop(columns=["ahi"])
y_alt = convert_ahi(df_fa_alt["ahi"])

X_train_alt, X_test_alt, y_train_alt, y_test_alt = train_test_split(
    X_alt, y_alt, test_size=0.2, random_state=42
)

fa_transformer_alt.fit(X_train_alt)
X_train_fa_alt = fa_transformer_alt.transform(X_train_alt)
X_test_fa_alt = fa_transformer_alt.transform(X_test_alt)

X_alt = fa_transformer_alt.transform(X_alt)

df_fa_alt = pd.DataFrame(X_alt, columns=[f"Factor_{i+1}" for i in range(X_alt.shape[1])])
df_fa_alt["ahi"] = y_alt.values

In [33]:
""" 
key:
alt = dropping ess and isi before factor analysis
var = varimax rotation instead of promax
v1 = raw data
v2 = factor analysis (default promax)
"""

fa_transformer_var_alt = Factor_Analyzer_Transformer(n_factors=18, rotation="varimax")

df_fa_var_alt = df.copy()
df_fa_var_alt.drop(columns=['ess_total_score', 'isi_total_score'], inplace=True)

X_var_alt = df_fa_var_alt.drop(columns=["ahi"])
y_var_alt = df_fa_var_alt["ahi"]

X_train_var_alt, X_test_var_alt, y_train_var_alt, y_test_var_alt = train_test_split(
    X_var_alt, y_var_alt, test_size=0.2, random_state=42
)

fa_transformer_var_alt.fit(X_train_var_alt)

X_var_alt = fa_transformer_var_alt.transform(X_var_alt)

df_fa_var_alt = pd.DataFrame(X_var_alt, columns=[f"Factor_{i+1}" for i in range(X_var_alt.shape[1])])
df_fa_var_alt["ahi"] = y_var_alt.values

In [ ]:
""" 
putting the 4 variations of the dataset into mlflow with metadata and tags for versioning
"""

tags_default = {"initial_version": "raw_data"}
dataset_v1 = create_versioned_dataset(df, version=1, tags=tags_default)

v2_tags = {"initial_version": "raw_data", "modification": "promax factor analysis features"}
dataset_v2 = create_versioned_dataset(df_fa, version=2, tags=v2_tags)

v2_alt_tags = {"initial_version": "raw_data", "modification": "promax factor analysis features with ess and isi dropped"}
dataset_v2_alt = create_versioned_dataset(df_fa_alt, version=2.1, tags=v2_alt_tags)

v2_var_tags = {"initial_version": "raw_data", "modification": "varimax factor analysis features"}
dataset_v2_var = create_versioned_dataset(df_fa_var, version=2.2, tags=v2_var_tags)

v2_alt_var_tags = {"initial_version": "raw_data", "modification": "varimax factor analysis features with ess and isi dropped"}
dataset_v2_alt_var = create_versioned_dataset(df_fa_var_alt, version=2.3, tags=v2_alt_var_tags)

/Users/jack/Repos/apnea-predictor/conda_env/lib/python3.14/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
/Users/jack/Repos/apnea-predictor/conda_env/lib/python3.14/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Inte